# Election-sector hedge — data-sourced version

Same pitch as `solar.ipynb`, but **inputs are pulled from the web** at run time:

| Input | Source |
|-------|--------|
| TAN prices & returns | [Yahoo Finance](https://finance.yahoo.com/quote/TAN/history/) via `yfinance` |
| Trump-win contract metadata | [Polymarket Gamma API](https://gamma-api.polymarket.com) |
| Trump-win price history | [Polymarket CLOB API](https://clob.polymarket.com/prices-history) |
| Harris-win counterfactual | Estimated from pre-election regression of TAN on Polymarket moves |
| Residual vol | Std dev of regression residuals (pre-election window) |
| Bid-ask spread | `orderPriceMinTickSize` from Gamma × 10 ticks (documented fallback) |

**Episode:** \$100k in Invesco Solar ETF (TAN), hedged with a Polymarket Trump-win contract around the 2024 US election.

In [ ]:
import json
import urllib.request
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

%matplotlib inline

GAMMA = "https://gamma-api.polymarket.com"
CLOB = "https://clob.polymarket.com"
MARKET_SLUG = "will-donald-trump-win-the-2024-us-presidential-election"
ELECTION_EVE = pd.Timestamp("2024-11-04")
EVENT_DAY = pd.Timestamp("2024-11-06")
V0 = 100_000
N_MC = 200_000
RNG = np.random.default_rng(7)


def fetch_json(url: str):
    req = urllib.request.Request(url, headers={"User-Agent": "paris-hack/hedgingresearch"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.load(resp)

## 1. Fetch Polymarket contract

In [ ]:
market = fetch_json(f"{GAMMA}/markets?slug={MARKET_SLUG}&closed=true")[0]
token_ids = json.loads(market["clobTokenIds"])
outcomes = json.loads(market["outcomes"])
yes_idx = outcomes.index("Yes")
yes_token = token_ids[yes_idx]
tick_size = float(market["orderPriceMinTickSize"])

print("Market:", market["question"])
yes_settled = float(json.loads(market["outcomePrices"])[yes_idx]) >= 0.99
print("Resolved:", market.get("closedTime"), "→", "Yes" if yes_settled else "No")
print("Min tick:", tick_size)
print("Yes token id:", yes_token[:20] + "…")

## 2. Fetch Trump-win price history (daily)

In [ ]:
history = fetch_json(
    f"{CLOB}/prices-history?market={yes_token}&interval=max&fidelity=1440"
)["history"]

poly = pd.DataFrame(history)
poly["date"] = (
    pd.to_datetime(poly["t"], unit="s", utc=True)
    .dt.tz_convert("America/New_York")
    .apply(lambda ts: ts.date())
)
trump_prob = poly.groupby("date")["p"].last().sort_index().rename("trump_prob")
trump_prob.index = pd.to_datetime(trump_prob.index)

print(f"Polymarket history: {trump_prob.index.min().date()} → {trump_prob.index.max().date()} ({len(trump_prob)} days)")
print("\nLast week before election:")
display(trump_prob.loc["2024-10-28":"2024-11-05"].to_frame())

## 3. Fetch TAN prices

In [ ]:
tan_close = yf.download("TAN", start="2024-09-01", end="2024-11-15", progress=False, auto_adjust=True)["Close"]
if isinstance(tan_close, pd.DataFrame):
    tan_close = tan_close.iloc[:, 0]
tan_close.index = pd.to_datetime(tan_close.index).normalize()
tan_ret = tan_close.pct_change().rename("tan_ret")

print("TAN around election:")
display(pd.DataFrame({"close": tan_close, "return": tan_ret}).loc["2024-11-01":"2024-11-08"])

## 4. Derive model parameters from data

In [ ]:
# --- [REAL] election-eve implied P(Trump) from Polymarket close on Nov 4 ---
p_event = float(trump_prob.asof(ELECTION_EVE))
yes_mid = p_event

# --- [REAL] realized TAN return on result day (Nov 5 close → Nov 6 close) ---
ret_if_event = float(tan_ret.loc[EVENT_DAY])

# --- [EST] Harris counterfactual: regress TAN daily returns on ΔTrump prob (pre-election) ---
panel = pd.concat([tan_ret, trump_prob], axis=1).dropna()
panel["d_trump_prob"] = panel["trump_prob"].diff()
reg_window = panel.loc["2024-09-01":ELECTION_EVE].dropna()

beta, intercept = np.polyfit(reg_window["d_trump_prob"], reg_window["tan_ret"], 1)
residuals = reg_window["tan_ret"] - (intercept + beta * reg_window["d_trump_prob"])
ret_if_not_reg = beta * (0.0 - p_event)

# Alternative: linear extrapolation from realized move (Trump prob  p_eve → 1)
beta_event = ret_if_event / (1.0 - p_event)
ret_if_not_extrap = beta_event * (0.0 - p_event)

# Use regression estimate as primary; keep extrapolation for comparison
ret_if_not = ret_if_not_reg

# --- [EST] residual vol from regression residuals ---
resid_sigma = float(residuals.std())

# --- [EST] spread: 10 ticks (Gamma min tick is 0.001 → 1¢) ---
spread = tick_size * 10
yes_cost = yes_mid + spread

sources = pd.DataFrame([
    {"parameter": "p_event / yes_mid", "value": f"{p_event:.4f}", "source": f"Polymarket Yes close on {ELECTION_EVE.date()}"},
    {"parameter": "ret_if_event", "value": f"{ret_if_event:+.2%}", "source": f"TAN close-to-close on {EVENT_DAY.date()} (yfinance)"},
    {"parameter": "ret_if_not (regression)", "value": f"{ret_if_not_reg:+.2%}", "source": f"β={beta:.2f} × (0 − {p_event:.2f}), n={len(reg_window)} days Sep–Nov"},
    {"parameter": "ret_if_not (extrapolation)", "value": f"{ret_if_not_extrap:+.2%}", "source": f"Scale Nov-6 move across ({p_event:.2f} → 0)"},
    {"parameter": "resid_sigma", "value": f"{resid_sigma:.2%}", "source": "Std dev of regression residuals"},
    {"parameter": "spread", "value": f"{spread:.3f}", "source": f"10 × Gamma tick size ({tick_size})"},
])
display(sources)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(trump_prob.index, trump_prob.values, color="#8e44ad", lw=1.5)
axes[0].axvline(ELECTION_EVE, color="k", ls="--", lw=0.8, label="Election eve")
axes[0].set_title("Polymarket: P(Trump win)")
axes[0].set_ylabel("Probability")
axes[0].legend()

axes[1].plot(tan_close.loc["2024-10-01":"2024-11-14"], color="#e67e22", lw=1.5)
axes[1].axvline(EVENT_DAY, color="k", ls="--", lw=0.8, label="Result day")
axes[1].set_title("TAN adjusted close")
axes[1].set_ylabel("USD")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Variance-minimising hedge

In [ ]:
Vw = V0 * (1 + ret_if_event)
Vl = V0 * (1 + ret_if_not)
N = Vl - Vw
premium = N * yes_cost

wealth_win_hedged = Vw + N * (1 - yes_cost)
wealth_lose_hedged = Vl - N * yes_cost

print(f"Hedge size: {N:,.0f} YES contracts @ ${yes_cost:.3f}  (premium ${premium:,.0f})")
print(f"Locked wealth (two-state): ${wealth_win_hedged:,.0f} / ${wealth_lose_hedged:,.0f}")

## 6. Monte Carlo with residual (basis) risk

In [ ]:
def mean_std_two_state(win_val, lose_val, p):
    m = p * win_val + (1 - p) * lose_val
    v = p * (win_val - m) ** 2 + (1 - p) * (lose_val - m) ** 2
    return m, np.sqrt(v)


def pct(x):
    return 100 * x / V0


m_unh, sd_unh = mean_std_two_state(Vw, Vl, p_event)
m_hed, sd_hed = mean_std_two_state(wealth_win_hedged, wealth_lose_hedged, p_event)

event = RNG.random(N_MC) < p_event
base_ret = np.where(event, ret_if_event, ret_if_not)
noise = RNG.normal(0, resid_sigma, N_MC)
tan_sim = base_ret + noise

port_unh = V0 * (1 + tan_sim)
hedge_payoff = np.where(event, N, 0.0) - premium
port_hed = port_unh + hedge_payoff

pnl_unh = port_unh - V0
pnl_hed = port_hed - V0

var_removed = 1 - np.var(pnl_hed) / np.var(pnl_unh)
hedge_cost_usd = pnl_unh.mean() - pnl_hed.mean()
hedge_cost_bps = 1e4 * hedge_cost_usd / V0
var5_unh = np.percentile(pnl_unh, 5)
var5_hed = np.percentile(pnl_hed, 5)

## 7. Summary

In [ ]:
print("=" * 60)
print("ELECTION-SECTOR HEDGE — $100k TAN vs Trump-win contract")
print("(parameters sourced from Polymarket + yfinance)")
print("=" * 60)
print(f"Market-implied P(event)        : {p_event:.1%}  [Polymarket {ELECTION_EVE.date()}]")
print(f"TAN if event / if not          : {ret_if_event:+.1%} / {ret_if_not:+.1%}")
print(f"  (alt extrapolation if not)   : {ret_if_not_extrap:+.1%}")
print(f"Contract bought at             : ${yes_cost:.3f}  (mid ${yes_mid:.3f} + {spread*100:.1f}c spread)")
print(f"Hedge size                     : {N:,.0f} contracts  (premium ${premium:,.0f})")
print("-" * 60)
print("TWO-STATE (election-driven only)")
print(f"  Unhedged  E=${m_unh:,.0f}   sd=${sd_unh:,.0f}  ({pct(sd_unh):.1f}% of notional)")
print(f"  Hedged    E=${m_hed:,.0f}   sd=${sd_hed:,.0f}  (locked across states)")
print("-" * 60)
print("FULL MODEL (with residual / basis risk)")
print(f"  Unhedged P&L : mean ${pnl_unh.mean():,.0f}  sd ${pnl_unh.std():,.0f} ({pct(pnl_unh.std()):.1f}%)")
print(f"  Hedged   P&L : mean ${pnl_hed.mean():,.0f}  sd ${pnl_hed.std():,.0f} ({pct(pnl_hed.std()):.1f}%)")
print(f"  >> VARIANCE REMOVED        : {var_removed:.1%}")
print(f"  >> RESIDUAL (basis) risk   : {1 - var_removed:.1%}  (irreducible)")
print(f"  >> HEDGE COST              : ${hedge_cost_usd:,.0f}  ({hedge_cost_bps:.0f} bps of notional)")
print("-" * 60)
print("TAIL (5% worst case)")
print(f"  Unhedged  : {pct(var5_unh):+.1f}%   (${var5_unh:,.0f})")
print(f"  Hedged    : {pct(var5_hed):+.1f}%   (${var5_hed:,.0f})")
print(f"  Drawdown cut by ~{pct(var5_hed - var5_unh):+.1f} pts in the tail")
print("=" * 60)

## 8. Chart

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))

bins = np.linspace(pct(pnl_unh.min()), pct(pnl_unh.max()), 80)
ax[0].hist(pct(pnl_unh), bins=bins, alpha=0.55, label="Unhedged", color="#c0392b")
ax[0].hist(pct(pnl_hed), bins=bins, alpha=0.75, label="Hedged", color="#16a085")
ax[0].axvline(0, color="k", lw=0.8)
ax[0].set_title("P&L distribution (% of notional)")
ax[0].set_xlabel("Return %")
ax[0].set_ylabel("Frequency")
ax[0].legend()

labels = ["Unhedged", "Hedged"]
sds = [pct(pnl_unh.std()), pct(pnl_hed.std())]
tails = [pct(var5_unh), pct(var5_hed)]
x = np.arange(2)
ax[1].bar(x - 0.2, sds, 0.4, label="Volatility (%)", color="#2980b9")
ax[1].bar(x + 0.2, [-t for t in tails], 0.4, label="Tail loss, 5% (%)", color="#e67e22")
ax[1].set_xticks(x)
ax[1].set_xticklabels(labels)
ax[1].set_title(f"Risk: {var_removed:.0%} of variance removed for ~{hedge_cost_bps:.0f} bps")
ax[1].legend()

plt.tight_layout()
plt.savefig("hedge_chart_data.png", dpi=140)
plt.show()
print("chart saved to hedge_chart_data.png")